In [1]:
import numpy as np
from PIL import Image


In [2]:
def replace_color(
    input_path: str,
    output_path: str,
    target_color: tuple[int, int, int] | tuple[int, int, int, int],
    replacement_color: tuple[int, int, int, int],
    tolerance: int = 0,
) -> None:
    """Replace target color occurrences with a replacement color in an image.

    Converts the source image to RGBA, identifies pixels matching
    target_color within an optional Euclidean Manhattan channel
    tolerance, substitutes matching pixels with replacement_color, and
    writes the result as a PNG.

    Args:
        input_path: Filesystem path to the source image file.
        output_path: Destination path where the modified PNG is written.
        target_color: RGB or RGBA color tuple to replace, with integer
            channel values in [0, 255].
        replacement_color: RGBA color tuple to substitute, with integer
            channel values in [0, 255].
        tolerance: Maximum absolute per-channel difference for pixel
            matching. Defaults to 0 (exact match).

    Returns:
        None. Saves the modified image directly to disk.

    Example:
        >>> replace_color(
        ...     "input.png",
        ...     "output.png",
        ...     target_color=(255, 255, 255),
        ...     replacement_color=(26, 118, 249, 255),
        ...     tolerance=10,
        ... )
    """
    # Ensure RGBA mode so transparency channel operations are preserved
    img = Image.open(input_path).convert("RGBA")
    data = np.array(img)

    r, g, b = data[..., 0], data[..., 1], data[..., 2]
    tr, tg, tb = target_color[:3]

    if tolerance == 0:
        match_mask = (r == tr) & (g == tg) & (b == tb)
    else:
        # Match within an absolute per-channel window for anti-aliased edges
        match_mask = (
            (np.abs(r.astype(int) - tr) <= tolerance)
            & (np.abs(g.astype(int) - tg) <= tolerance)
            & (np.abs(b.astype(int) - tb) <= tolerance)
        )

    # Match target alpha when explicitly supplied
    if len(target_color) == 4:
        match_mask = match_mask & (data[..., 3] == target_color[3])

    data[match_mask] = replacement_color

    output_img = Image.fromarray(data, mode="RGBA")
    output_img.save(output_path, format="PNG")


In [7]:
def hex_to_rgba(
    hex_code: str, default_alpha: int = 255
) -> tuple[int, int, int, int]:
    """Convert a hex color string into an RGBA integer tuple.

    Supports 6-character (#RRGGBB) and 8-character (#RRGGBBAA) hex
    notations with an optional leading hash symbol.

    Args:
        hex_code: Hex color string in '#RRGGBB' or '#RRGGBBAA' format.
        default_alpha: Alpha channel value in [0, 255] applied when
            hex_code has 6 hex digits. Defaults to 255 (fully opaque).

    Returns:
        An (R, G, B, A) tuple of integer channel values in [0, 255].

    Raises:
        ValueError: If hex_code does not contain 6 or 8 hex digits.

    Example:
        >>> hex_to_rgba("#1A76F9")
        (26, 118, 249, 255)
        >>> hex_to_rgba("FFFFFF80")
        (255, 255, 255, 128)
    """
    hex_code = hex_code.lstrip("#")
    if len(hex_code) == 6:
        r, g, b = (int(hex_code[i : i + 2], 16) for i in (0, 2, 4))
        return (r, g, b, default_alpha)
    elif len(hex_code) == 8:
        return tuple(int(hex_code[i : i + 2], 16) for i in (0, 2, 4, 6))
    raise ValueError(f"Invalid hex color format: {hex_code}")


def replace_hex_color(
    input_path: str,
    output_path: str,
    target_hex: str = "#000000",
    replacement_hex: str = "#1A76F9",
    tolerance: int = 0,
) -> None:
    """Replace occurrences of a target hex color with a replacement hex color.

    Parses target and replacement hex strings into RGBA values, locates
    matching pixels in the converted RGBA image within an optional
    per-channel tolerance, and writes the output as a PNG.

    Args:
        input_path: Filesystem path to the input image file.
        output_path: Destination path where the modified PNG is written.
        target_hex: Hex code string for the color to replace. Defaults to
            '#000000'.
        replacement_hex: Hex code string for the replacement color.
            Defaults to '#1A76F9'.
        tolerance: Maximum absolute per-channel difference for pixel
            matching. Defaults to 0 (exact match).

    Returns:
        None. Saves the modified image directly to disk.

    Raises:
        ValueError: If target_hex or replacement_hex cannot be parsed as
            a 6- or 8-digit hex code.

    Example:
        >>> replace_hex_color(
        ...     "icon.png",
        ...     "icon_blue.png",
        ...     target_hex="#FFFFFF",
        ...     replacement_hex="#1A76F9",
        ...     tolerance=15,
        ... )
    """
    target_rgba = hex_to_rgba(target_hex)
    replacement_rgba = hex_to_rgba(replacement_hex)

    img = Image.open(input_path).convert("RGBA")
    data = np.array(img)

    r, g, b = data[..., 0], data[..., 1], data[..., 2]
    tr, tg, tb = target_rgba[:3]

    if tolerance == 0:
        match_mask = (r == tr) & (g == tg) & (b == tb)
    else:
        # Match within an absolute per-channel window for anti-aliased edges
        match_mask = (
            (np.abs(r.astype(int) - tr) <= tolerance)
            & (np.abs(g.astype(int) - tg) <= tolerance)
            & (np.abs(b.astype(int) - tb) <= tolerance)
        )

    # Respect source alpha if the target specifies an explicit 8-digit alpha
    if len(target_hex.lstrip("#")) == 8:
        match_mask = match_mask & (data[..., 3] == target_rgba[3])

    data[match_mask] = replacement_rgba

    output_img = Image.fromarray(data, mode="RGBA")
    output_img.save(output_path, format="PNG")


In [8]:
input_locations = ["/home/david/Downloads/scales.drawio.png", "/home/david/Downloads/checklist.drawio.png", "/home/david/Downloads/book.drawio.png", "/home/david/Downloads/spyglass.drawio.png", "/home/david/Downloads/clusters.drawio.png"]

output_locations = ["/home/david/Git/SDLT_Number_Crunching/Images/scales.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/checklist.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/book.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/spyglass.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/clusters.drawio.png"]

WHITE = (255, 255, 255)
TRANSPARENT = (0, 0, 0, 0)

for location in output_locations:

    # # Set tolerance=0 for exact white, or e.g. tolerance=10-20 to clean up near-white compression artifacts
    # replace_color(
    #     input_path=location,
    #     output_path=location,
    #     target_color=(0, 0, 0),
    #     replacement_color=(26, 118, 249),
    #     tolerance=5,
    # )


    replace_hex_color(
        input_path=location,
        output_path=location,
        target_hex="#FFFFFF",
        replacement_hex="#1A76F9",
        tolerance=15,  # Catches dark anti-aliased edges often found around black text/shapes
    )